In [ ]:
import pandas as pd
import requests
import time


In [ ]:
df = pd.read_csv("uber.csv")

print(df.head())

In [ ]:
# remove extra index column
df = df.drop(columns=["Unnamed: 0"])

# convert datetime
df['pickup_datetime'] = pd.to_datetime(
    df['pickup_datetime'],
    utc=True
)

In [ ]:
# create hourly timestamp
df['pickup_hour'] = df['pickup_datetime'].dt.floor('h')

# reduce API calls
df['lat_round'] = df['pickup_latitude'].round(2)
df['lon_round'] = df['pickup_longitude'].round(2)

weather_keys = df[['pickup_hour']].drop_duplicates()


print(len(weather_keys))

In [ ]:
def get_weather_fast(row):

    date = str(row.pickup_hour.date())
    hour = row.pickup_hour.hour

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": 40.7128,
        "longitude": -74.0060,
        "start_date": date,
        "end_date": date,
        "hourly":
        "temperature_2m,precipitation,windspeed_10m"
    }

    try:
        r = requests.get(url, params=params, timeout=10)
        data = r.json()

        return {
            "pickup_hour": row.pickup_hour,
            "temperature":
                data['hourly']['temperature_2m'][hour],
            "precipitation":
                data['hourly']['precipitation'][hour],
            "wind_speed":
                data['hourly']['windspeed_10m'][hour]
        }

    except:
        return None

In [ ]:
results = []

for _, row in weather_keys.iterrows():
    data = get_weather_fast(row)
    if data is not None:
        results.append(data)

print("Weather records collected:", len(results))

In [ ]:
def get_weather_fast(row):

    date = str(row.pickup_hour.date())

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": 40.7128,
        "longitude": -74.0060,
        "start_date": date,
        "end_date": date,
        "hourly":
        "temperature_2m,precipitation,windspeed_10m"
    }

    try:
        r = requests.get(url, params=params, timeout=10)
        data = r.json()

        return {
            "pickup_hour": row.pickup_hour,
            "temperature":
                data['hourly']['temperature_2m'][0],
            "precipitation":
                data['hourly']['precipitation'][0],
            "wind_speed":
                data['hourly']['windspeed_10m'][0]
        }

    except:
        return None

In [ ]:
df = df.sort_values('pickup_hour')

weather_cols = ['temperature', 'precipitation', 'wind_speed']

# create columns if merge failed
for col in weather_cols:
    if col not in df.columns:
        df[col] = pd.NA

df[weather_cols] = (
    df[weather_cols]
    .ffill()
    .bfill()
)

print("Missing values after fill:")
print(df[weather_cols].isna().sum())

In [ ]:
df = df.sort_values('pickup_hour')

df[['temperature','precipitation','wind_speed']] = (
    df[['temperature','precipitation','wind_speed']]
    .ffill()
    .bfill()
)

print("Missing values after fill:")
print(df[['temperature','precipitation','wind_speed']].isna().sum())

In [ ]:
print(df.columns)
print(df[['temperature','precipitation','wind_speed']].head())

In [ ]:
df.to_csv("uber_weather_enriched.csv", index=False)